# REA Colab Quickstart

Clean quickstart for running the full-sequence SMC and mixture MCMC experiments from a fresh Colab runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/YOUR_USER/YOUR_REPO.git'  # edit this
REPO_DIR = '/content/rea_of_llms_supplementary_code-main'

import os, subprocess, pathlib
if pathlib.Path(REPO_DIR).exists():
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('cwd =', os.getcwd())

In [ ]:
!pip install -r requirements-colab.txt

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
!python full_sequence_smc_experiment/run_smc_sampler.py \
  --device auto \
  --num-particles 2 \
  --max-new-tokens 2 \
  --lambdas 0,0 \
  --mcmc-steps-per-level 1 \
  --output-dir /content/drive/MyDrive/rea_outputs/full_sequence_smoke

In [ ]:
!python mixture_mcmc_experiment/run_experiment.py \
  --device auto \
  --num-iterations 2 \
  --burn-in 0 \
  --thinning 1 \
  --max-new-tokens 2 \
  --lambdas=-0.5,0.0 \
  --output-dir /content/drive/MyDrive/rea_outputs/mixture_smoke

## Publish a selected run

Only do this for a run you want reviewed on GitHub. The command appends to `results/EXPERIMENT_LOG.md` and copies only `summary.json`, `report.md`, and a small `samples_preview.jsonl`.

In [ ]:
!python mixture_mcmc_experiment/run_experiment.py \
  --device auto \
  --run-name colab_mixture_selected_example \
  --num-iterations 5 \
  --burn-in 0 \
  --thinning 1 \
  --max-new-tokens 5 \
  --lambdas=-0.5,0.0 \
  --output-dir /content/drive/MyDrive/rea_outputs/colab_mixture_selected_example \
  --experiment-log results/EXPERIMENT_LOG.md \
  --publish-result-dir results/selected_runs

In [ ]:
!git status --short
!git add results/EXPERIMENT_LOG.md results/selected_runs README.md requirements-colab.txt pyproject.toml rea_llms full_sequence_smc_experiment mixture_mcmc_experiment notebooks tests
!git commit -m "Add selected Colab run" || true
!git push